In [ ]:
from google.colab import drive
import sys

drive.mount("/content/drive")
sys.path.append("/content/drive/MyDrive/colab_env/lib/python3.11/site-packages")

Mounted at /content/drive


In [ ]:
import tensorflow as tf
print('TF: {}'.format(tf.__version__))

import tensorflow_transform as tft
print('Transform: {}'.format(tft.__version__))

TF: 2.18.0
Transform: 1.16.0


In [ ]:
tft_output = tft.TFTransformOutput("/content/drive/MyDrive/DATA/train-transform-fn/transform_fn")
original_feature_spec = tft_output.transformed_feature_spec()
label_key = 'scaled_price'
original_feature_spec

transformed_feature_spec = original_feature_spec.copy()
transformed_feature_spec.pop(label_key)  #
transformed_feature_spec

{'company_xf': FixedLenFeature(shape=[], dtype=tf.int64, default_value=None),
 'cpu_brand_xf': FixedLenFeature(shape=[], dtype=tf.int64, default_value=None),
 'cpu_family_xf': FixedLenFeature(shape=[], dtype=tf.int64, default_value=None),
 'gpu_brand_xf': FixedLenFeature(shape=[], dtype=tf.int64, default_value=None),
 'gpu_model_xf': FixedLenFeature(shape=[], dtype=tf.int64, default_value=None),
 'has_hdd': FixedLenFeature(shape=[], dtype=tf.float32, default_value=None),
 'has_ssd': FixedLenFeature(shape=[], dtype=tf.float32, default_value=None),
 'inches_bucket': FixedLenFeature(shape=[], dtype=tf.float32, default_value=None),
 'opsys_xf': FixedLenFeature(shape=[], dtype=tf.int64, default_value=None),
 'scaled_height': FixedLenFeature(shape=[], dtype=tf.float32, default_value=None),
 'scaled_memory_size': FixedLenFeature(shape=[], dtype=tf.float32, default_value=None),
 'scaled_ram': FixedLenFeature(shape=[], dtype=tf.float32, default_value=None),
 'scaled_weight': FixedLenFeature(sha

In [ ]:
def parse_fn(example_proto):
    parsed = tf.io.parse_single_example(example_proto, transformed_feature_spec)
    label = parsed.pop(label_key)
    return parsed, label

In [ ]:
def load_dataset_from_gz(tfrecord_path, batch_size=128, shuffle=False):
    dataset = tf.data.TFRecordDataset(tfrecord_path, compression_type="GZIP")
    dataset = dataset.map(parse_fn, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        dataset = dataset.shuffle(buffer_size=10000)

    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
from tensorflow import keras

train_ds = load_dataset_from_gz("/content/drive/MyDrive/DATA/train-00000-of-00001.gz", shuffle=True)
eval_ds = load_dataset_from_gz("/content/drive/MyDrive/DATA/eval-00000-of-00001.gz")

In [ ]:
train_ds

<_PrefetchDataset element_spec=({'company_xf': TensorSpec(shape=(None,), dtype=tf.int64, name=None), 'cpu_brand_xf': TensorSpec(shape=(None,), dtype=tf.int64, name=None), 'cpu_family_xf': TensorSpec(shape=(None,), dtype=tf.int64, name=None), 'gpu_brand_xf': TensorSpec(shape=(None,), dtype=tf.int64, name=None), 'gpu_model_xf': TensorSpec(shape=(None,), dtype=tf.int64, name=None), 'has_hdd': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'has_ssd': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'inches_bucket': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'opsys_xf': TensorSpec(shape=(None,), dtype=tf.int64, name=None), 'scaled_height': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'scaled_memory_size': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'scaled_ram': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'scaled_weight': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'scaled_width': TensorSpec(shape=(None,), dtype=tf.

In [ ]:
inputs = {
    key: tf.keras.Input(shape=val.shape, name=key, dtype=val.dtype)
    for key, val in transformed_feature_spec.items()
    if key != 'scaled_price'
}

# Cast all inputs to float32 to match dtype for concatenation
casted_inputs = [
    tf.keras.layers.Lambda(lambda x: tf.expand_dims(tf.cast(x, tf.float32), -1))(inp)
    for inp in inputs.values()
]

# Flatten and concatenate all features
concatenated = keras.layers.Concatenate()(casted_inputs)
x = keras.layers.Dense(128, activation='relu')(concatenated)
x = keras.layers.Dense(64, activation='relu')(x)
x = keras.layers.Dense(32, activation='relu')(x)
output = keras.layers.Dense(1)(x)

model = keras.Model(inputs=inputs, outputs=output)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=["mean_squared_error"],
    metrics=[["mean_absolute_error"], ["accuracy"]]
)

model.summary()

# for features, labels in eval_ds.take(1):
#     print(f"{len(labels)}")

#model.fit(train_ds, validation_data=eval_ds, epochs=30)

Model: "model_13"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 company_xf (InputLayer)     [(None,)]                    0         []                            
                                                                                                  
 cpu_brand_xf (InputLayer)   [(None,)]                    0         []                            
                                                                                                  
 cpu_family_xf (InputLayer)  [(None,)]                    0         []                            
                                                                                                  
 gpu_brand_xf (InputLayer)   [(None,)]                    0         []                            
                                                                                           

In [ ]:
for x_batch, y_batch in eval_ds.take(1):  # assuming tf.data.Dataset
    y_pred = model.predict(x_batch)
    print("True scaled prices:", y_batch.numpy()[:5])
    print("Predicted scaled prices:", y_pred[:5].flatten())

4/4 [==============================] - 0s 6ms/step
True scaled prices: [ 1.5684174  -1.0852911   0.79021066  0.66302186  0.7575455 ]
Predicted scaled prices: [-0.17687348 -0.12255824 -1.1678418   0.02609357 -0.8304092 ]


In [ ]:
for x_batch, y_batch in eval_ds.take(1):
    for key in x_batch:
        if 'scaled' in key:
            print(f"{key}: mean={x_batch[key].numpy().mean():.4f}, std={x_batch[key].numpy().std():.4f}")

scaled_height: mean=-0.0385, std=1.0082
scaled_memory_size: mean=0.0989, std=1.0826
scaled_ram: mean=0.0445, std=0.9842
scaled_weight: mean=0.0324, std=1.0309
scaled_width: mean=-0.0385, std=1.0082


In [ ]:
tiny_ds = train_ds.take(1).repeat(100)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

model.fit(tiny_ds, epochs=20)

Epoch 1/20
100/100 [==============================] - 65s 628ms/step - loss: 1.0185 - mae: 0.8691
Epoch 2/20
100/100 [==============================] - 62s 616ms/step - loss: 0.9927 - mae: 0.8565
Epoch 3/20
100/100 [==============================] - 61s 614ms/step - loss: 0.9850 - mae: 0.8535
Epoch 4/20
100/100 [==============================] - 58s 582ms/step - loss: 0.9811 - mae: 0.8525
Epoch 5/20
100/100 [==============================] - 55s 545ms/step - loss: 0.9637 - mae: 0.8437
Epoch 6/20
100/100 [==============================] - 60s 598ms/step - loss: 0.9729 - mae: 0.8492
Epoch 7/20
100/100 [==============================] - 60s 599ms/step - loss: 0.9722 - mae: 0.8476
Epoch 8/20
100/100 [==============================] - 61s 610ms/step - loss: 0.9828 - mae: 0.8505
Epoch 9/20
100/100 [==============================] - 58s 584ms/step - loss: 0.9648 - mae: 0.8448
Epoch 10/20
100/100 [==============================] - 61s 608ms/step - loss: 0.9387 - mae: 0.8293
Epoch 11/20
100/100

In [ ]:
for x_batch, y_batch in train_ds.take(1):
    print("Y true:", y_batch.numpy()[:5])
    for key in x_batch:
        print(f"{key}: {x_batch[key].numpy()[:5]}")

Y true: [ 0.56814975  0.44404563 -0.236011    0.22447562 -1.1485952 ]
company_xf: [4 0 2 5 6]
cpu_brand_xf: [0 1 1 0 1]
cpu_family_xf: [1 2 4 0 4]
gpu_brand_xf: [0 0 0 0 2]
gpu_model_xf: [0 0 0 3 2]
has_hdd: [0. 0. 0. 0. 0.]
has_ssd: [1. 1. 1. 1. 1.]
inches_bucket: [0. 0. 1. 2. 0.]
opsys_xf: [0 0 2 1 1]
scaled_height: [ 1.2780809  1.2780809  1.2780809 -1.2333826 -0.6704684]
scaled_memory_size: [ 1.7959111  -0.40378174 -0.7703972  -0.9537049  -0.7703972 ]
scaled_ram: [ 1.594451    1.594451    0.10830818 -1.0062989   0.10830818]
scaled_weight: [ 1.4132329  1.4132329  1.4132329 -0.6067458  1.4132329]
scaled_width: [ 1.2781786   1.2781786   1.2781786  -1.2331328  -0.67077774]
typename_xf: [2 0 0 1 1]


In [ ]:
model.fit(train_ds.take(1).repeat(1000), steps_per_epoch=10, epochs=50)

Epoch 1/50
10/10 [==============================] - 6s 534ms/step - loss: 1.0935 - mean_absolute_error: 0.8831 - accuracy: 0.0000e+00
Epoch 2/50
10/10 [==============================] - 7s 742ms/step - loss: 1.0127 - mean_absolute_error: 0.8659 - accuracy: 0.0000e+00
Epoch 3/50
10/10 [==============================] - 5s 538ms/step - loss: 1.0154 - mean_absolute_error: 0.8706 - accuracy: 0.0000e+00
Epoch 4/50
10/10 [==============================] - 6s 640ms/step - loss: 0.9697 - mean_absolute_error: 0.8457 - accuracy: 0.0000e+00
Epoch 5/50
10/10 [==============================] - 6s 577ms/step - loss: 1.0268 - mean_absolute_error: 0.8717 - accuracy: 0.0000e+00
Epoch 6/50
10/10 [==============================] - 5s 527ms/step - loss: 1.0213 - mean_absolute_error: 0.8732 - accuracy: 0.0000e+00
Epoch 7/50
10/10 [==============================] - 7s 742ms/step - loss: 0.9766 - mean_absolute_error: 0.8447 - accuracy: 0.0000e+00
Epoch 8/50
10/10 [==============================] - 5s 524ms/s

KeyboardInterrupt: 